In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

results_dir = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/ldbc_snb_sf3_full_fiben_format_clean"
)

agg = pd.read_csv(results_dir / "benchmark_aggregate_results.csv")

if "official_id" not in agg.columns:
    agg["official_id"] = agg["query_name"].str.extract(r"^(IC\d+|IS\d+|INS\d+)")

def query_group_from_official_id(oid):
    if str(oid).startswith("IC"):
        return "complex_read"
    if str(oid).startswith("IS"):
        return "short_read"
    if str(oid).startswith("INS"):
        return "insert"
    return "other"

if "query_group" not in agg.columns:
    agg["query_group"] = agg["official_id"].apply(query_group_from_official_id)

hot = agg[agg["run_phase"] == "hot"].copy()

rows = []

for query_name, grp in hot.groupby("query_name"):
    best_all = grp.loc[grp["p95_latency_ms"].idxmin()]

    activated = grp[grp["final_benchmark_group"] != "control"].copy()
    primary = grp[grp["final_benchmark_group"] == "primary"].copy()

    best_activated = activated.loc[activated["p95_latency_ms"].idxmin()]

    if len(primary) > 0:
        best_primary = primary.loc[primary["p95_latency_ms"].idxmin()]
        primary_regret = (
            best_primary["p95_latency_ms"] - best_all["p95_latency_ms"]
        ) / best_all["p95_latency_ms"]
    else:
        best_primary = None
        primary_regret = np.nan

    n_C = 10
    n_A = activated["candidate_id"].nunique()
    dsr = 1 - (n_A / n_C)

    rows.append({
        "scale_label": "sf3",
        "official_id": best_all["official_id"],
        "query_name": query_name,
        "query_group": best_all["query_group"],

        "n_tested_configs": grp["candidate_id"].nunique(),
        "n_activated_configs": n_A,
        "DSR": dsr,

        "best_config": best_all["g_class"],
        "best_group": best_all["final_benchmark_group"],
        "best_design_pattern": best_all["design_pattern"],
        "best_p95_ms": best_all["p95_latency_ms"],

        "top1_preserved_by_activated": (
            best_activated["candidate_id"] == best_all["candidate_id"]
        ),
        "activated_regret": (
            best_activated["p95_latency_ms"] - best_all["p95_latency_ms"]
        ) / best_all["p95_latency_ms"],

        "best_primary_config": None if best_primary is None else best_primary["g_class"],
        "best_primary_p95_ms": None if best_primary is None else best_primary["p95_latency_ms"],
        "primary_regret": primary_regret,
    })

sf3_analysis_df = (
    pd.DataFrame(rows)
    .sort_values(["query_group", "official_id"])
    .reset_index(drop=True)
)

display(sf3_analysis_df)

print("Average DSR:", sf3_analysis_df["DSR"].mean())
print("Top-1 preservation activated:", sf3_analysis_df["top1_preserved_by_activated"].mean())
print("Mean activated regret:", sf3_analysis_df["activated_regret"].mean())
print("Mean primary regret:", sf3_analysis_df["primary_regret"].dropna().mean())

print("\nBest group counts:")
print(sf3_analysis_df["best_group"].value_counts())

print("\nSecondary affected winners:")
display(
    sf3_analysis_df[
        sf3_analysis_df["best_group"] == "secondary_affected"
    ][
        [
            "official_id",
            "query_name",
            "best_config",
            "best_design_pattern",
            "best_p95_ms",
            "best_primary_config",
            "best_primary_p95_ms",
            "primary_regret",
        ]
    ]
)

print("\nControl winners:")
display(
    sf3_analysis_df[
        sf3_analysis_df["best_group"] == "control"
    ][
        [
            "official_id",
            "query_name",
            "best_config",
            "best_design_pattern",
            "best_p95_ms",
            "best_primary_config",
            "best_primary_p95_ms",
            "primary_regret",
            "activated_regret",
        ]
    ]
)

out_path = results_dir / "schemalens_reduction_analysis_hot.csv"
sf3_analysis_df.to_csv(out_path, index=False)

print("Saved:", out_path)

,scale_label,official_id,query_name,query_group,n_tested_configs,n_activated_configs,DSR,best_config,best_group,best_design_pattern,best_p95_ms,top1_preserved_by_activated,activated_regret,best_primary_config,best_primary_p95_ms,primary_regret
0,sf3,IC1,IC1_TransitiveFriendsWithName,complex_read,2,2,0.8,G3,primary,root_with_references_or_summaries,260.022248,True,0.0,G3,260.022248,0.000000
1,sf3,IC2,IC2_RecentMessagesByFriends,complex_read,2,2,0.8,G0,primary,root_with_references,36.105321,True,0.0,G0,36.105321,0.000000
2,sf3,IC3,IC3_FriendsAndFriendsOfFriendsInCountries,complex_read,4,4,0.6,G0,primary,root_with_references,205.565454,True,0.0,G0,205.565454,0.000000
3,sf3,IC4,IC4_NewTopics,complex_read,2,2,0.8,G0,primary,root_with_references,45.188472,True,0.0,G0,45.188472,0.000000
4,sf3,IC5,IC5_NewGroups,complex_read,6,6,0.4,G7,secondary_affected,containment_baseline,193.801405,True,0.0,G3,205.203150,0.058832
5,sf3,IC6,IC6_TagCoOccurrence,complex_read,2,2,0.8,G0,primary,root_with_references,253.192134,True,0.0,G0,253.192134,0.000000
6,sf3,IC7,IC7_RecentLikers,complex_read,4,4,0.6,G4,secondary_affected,explicit_edge_collection,8.036814,True,0.0,G0,9.727438,0.210360
7,sf3,INS1,INS1_AddPerson,insert,2,2,0.8,G3,primary,root_with_references_or_summaries,1.432566,True,0.0,G3,1.432566,0.000000
8,sf3,INS2,INS2_AddLikeToPost,insert,2,2,0.8,G6,primary,referenced_or_reverse_indexed_edges,1.630815,True,0.0,G6,1.630815,0.000000
9,sf3,INS3,INS3_AddLikeToComment,insert,2,2,0.8,G4,primary,explicit_edge_collection,1.312449,True,0.0,G4,1.312449,0.000000


Average DSR: 0.7136363636363637
Top-1 preservation activated: 1.0
Mean activated regret: 0.0
Mean primary regret: 0.08942852994888324

Best group counts:
best_group
primary               15
secondary_affected     7
Name: count, dtype: int64

Secondary affected winners:


,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret
4,IC5,IC5_NewGroups,G7,containment_baseline,193.801405,G3,205.203150,0.058832
6,IC7,IC7_RecentLikers,G4,explicit_edge_collection,8.036814,G0,9.727438,0.210360
12,INS6,INS6_AddPost,G7,containment_baseline,2.207324,G0,2.580780,0.169189
13,INS7,INS7_AddComment,G7,containment_baseline,1.241873,G0,1.406190,0.132314
16,IS2,IS2_RecentMessagesOfPerson,G0,root_with_references,2.711272,None,NaN,NaN
20,IS6,IS6_ForumOfMessage,G0,root_with_references,1.332350,G7,2.399269,0.800780
21,IS7,IS7_RepliesOfMessage,G7,containment_baseline,14.036818,G3,21.146798,0.506524



Control winners:


,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret,activated_regret


Saved: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/ldbc_snb_sf3_full_fiben_format_clean/schemalens_reduction_analysis_hot.csv
